# 🦖 Você fugiria de um T. rex? — render de cinema (Blender Cycles na GPU)

A mesma animação e as mesmas câmeras do Short, montadas no **Blender** e renderizadas no **Cycles**: luz global, sombras suaves, **desfoque de movimento**, profundidade de campo e os modelos do Poly Haven em resolução cheia.

**Como usar**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU** → Salvar.
2. Menu **Ambiente de execução → Executar tudo**.
3. Na célula 2 o Google pede para conectar o **Google Drive**: aceite (os quadros ficam salvos lá; se o Colab cair, é só rodar tudo de novo que ele **continua de onde parou**).
4. No fim, o vídeo `trex_cinema.mp4` aparece na tela e fica no Drive, na pasta `trex_render`.

A célula 6 renderiza **um quadro de teste** e mostra **quanto tempo o filme inteiro vai levar** com a qualidade escolhida. Se ficar longo demais para a sua sessão, reduza para `LARGURA, ALTURA = 720, 1280` na célula 1 (a montagem amplia para 1080x1920) e rode de novo. Para a qualidade máxima, use `"cinema"` e deixe rodar em várias sessões: ele sempre continua de onde parou.

In [ ]:
# 1) Escolhas
QUALIDADE = "rapido"      # "rapido": 32 amostras + limpeza de ruído da GPU (~2,5-4,5 h na T4)  |  "cinema": 96 amostras (~3x mais tempo; rode em várias sessões)
LARGURA, ALTURA = 1080, 1920   # 1080x1920 = Short em Full HD. (720x1280 deixa ~2x mais rápido; a montagem amplia para 1080x1920)
BRANCH = "ccr-e37b66ae-ouzs0h"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Google Drive: os quadros ficam salvos lá (o render continua de onde parou se o Colab cair)
from google.colab import drive
drive.mount('/content/drive')
import os
SAIDA = '/content/drive/MyDrive/trex_render'
QUADROS = f'{SAIDA}/quadros_{LARGURA}x{ALTURA}_{QUALIDADE}'
os.makedirs(QUADROS, exist_ok=True)
print('quadros em', QUADROS, '| já prontos:', len([f for f in os.listdir(QUADROS) if f.endswith('.jpg') and os.path.getsize(f'{QUADROS}/{f}') > 0]))

In [ ]:
# 3) Projeto (só as pastas usadas, ~50 MB) e Blender 4.2 LTS — com novas tentativas se a conexão cair
import os, shutil, subprocess, time
def sh(c): return subprocess.run(c, shell=True, text=True, capture_output=True)
URL = 'https://github.com/hrosone/mentoria-jarvis.git'
NECESSARIOS = ['videos/trex_corrida/colab/dados/trex_anim.glb', 'videos/trex_corrida/colab/dados/trilheiro_anim.glb', 'videos/trex_corrida/colab/dados/maratonista_anim.glb',
               'videos/trex_corrida/colab/dados/cenario.glb', 'videos/trex_corrida/colab/cena_blender.py', 'videos/trex_corrida/montar.py', 'videos/biblioteca3d/polyhaven/baixar.py']
for tentativa in range(1, 6):
    shutil.rmtree('/content/mj', ignore_errors=True)
    r = sh(f'git clone -q --depth 1 --filter=blob:none --no-checkout -b {BRANCH} {URL} /content/mj && cd /content/mj && '
           f'git sparse-checkout set videos/trex_corrida videos/biblioteca3d/polyhaven && git checkout -q {BRANCH}')
    if r.returncode == 0 and all(os.path.getsize(f'/content/mj/{f}') > 0 for f in NECESSARIOS if os.path.exists(f'/content/mj/{f}')) and all(os.path.exists(f'/content/mj/{f}') for f in NECESSARIOS):
        print('projeto baixado'); break
    print(f'tentativa {tentativa} falhou ({r.stderr.strip()[-160:]}); tentando de novo...'); time.sleep(5 * tentativa)
else:
    raise SystemExit('Não consegui baixar o projeto do GitHub (conexão caindo). Rode esta célula de novo daqui a pouco.')
os.chdir('/content')
if not os.path.exists('/content/blender/blender'):
    for tentativa in range(1, 6):
        r = sh('wget -q -c --tries=3 https://download.blender.org/release/Blender4.2/blender-4.2.23-linux-x64.tar.xz && tar xf blender-4.2.23-linux-x64.tar.xz && mv blender-4.2.23-linux-x64 blender')
        if r.returncode == 0: break
        print(f'Blender: tentativa {tentativa} falhou; tentando de novo...'); time.sleep(5)
sh('apt-get -qq install -y libxi6 libxkbcommon0 libxrender1 libsm6 libgl1 libxxf86vm1 libxfixes3')
print(sh('/content/blender/blender --version').stdout.splitlines()[0])

In [ ]:
# 4) Modelos, texturas e céu do Poly Haven (CC0), na resolução cheia — com novas tentativas
import json, os, subprocess
os.chdir('/content/mj/videos/biblioteca3d/polyhaven')
def baixar(*a):
    for k in range(5):
        r = subprocess.run(['python3', 'baixar.py', *a], capture_output=True, text=True)
        if r.returncode == 0: return
        print('  tentando de novo:', ' '.join(a), '|', (r.stderr or r.stdout).strip()[-150:])
    raise SystemExit(f'Falhou ao baixar {a}: rode esta célula de novo (o que já baixou fica guardado).')
ids = list(json.load(open('/content/mj/videos/trex_corrida/colab/dados/instancias.json')).keys())
for i in ids: baixar('modelo', i, '--res', '1k'); print('modelo', i)
for t in ['forest_leaves_02', 'brown_mud_03', 'mossy_rock']: baixar('textura', t, '--res', '2k'); print('textura', t)
baixar('hdri', 'mossy_forest', '--res', '2k'); print('céu mossy_forest')
faltam = [i for i in ids if not os.path.exists(f'modelo/{i}/{i}_1k.gltf')]
print('tudo baixado' if not faltam else f'FALTAM: {faltam}')

In [ ]:
# 5) Monta a cena no Blender (atores animados, câmeras, mata, sol, céu, névoa)
%cd /content/mj/videos/trex_corrida/colab
!/content/blender/blender -b -P cena_blender.py -- --dados dados --ph ../../biblioteca3d/polyhaven --saida /content/cena.blend --qualidade {QUALIDADE} --larg {LARGURA} --alt {ALTURA} --gpu --quadros {QUADROS} 2>&1 | grep -E "INSTANCIAS|CORTES|PRONTO|FALTA|Error|rror:" 

In [ ]:
# 6) Quadro de teste (o plano em que ele corre na estrada com o T. rex atrás) e a previsão do tempo total
import time, subprocess
from IPython.display import Image, display
t0 = time.time()
subprocess.run(['/content/blender/blender', '-b', '/content/cena.blend', '-P', 'gpu.py', '-o', '/content/teste_#####', '-F', 'JPEG', '-f', '150'], capture_output=True)
seg = time.time() - t0
display(Image('/content/teste_00150.jpg', width=360))
print(f'1 quadro: {seg:.0f} s  ->  filme inteiro (1080 quadros): ~{seg * 1080 / 3600:.1f} h  (o 1º quadro inclui carregar a cena; os seguintes são mais rápidos)')

In [ ]:
# 7) Render do filme inteiro (retoma de onde parou; os quadros vão para o Drive)
import glob, os, subprocess, time
for f in glob.glob(f'{QUADROS}/*.jpg'):
    if os.path.getsize(f) == 0: os.remove(f)          # quadro que ficou pela metade quando o Colab caiu
p = subprocess.Popen(['/content/blender/blender', '-b', '/content/cena.blend', '-P', 'gpu.py', '-a'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
t0, feitos = time.time(), 0
for linha in p.stdout:
    if linha.startswith('Saved:'):
        feitos += 1
        if feitos % 10 == 0:
            prontos = len(glob.glob(f'{QUADROS}/*.jpg')); s = (time.time() - t0) / feitos
            print(f'{prontos}/1080 quadros  |  {s:.1f} s/quadro  |  falta ~{(1080 - prontos) * s / 60:.0f} min', flush=True)
    elif 'Error' in linha: print(linha.strip())
p.wait(); print('render terminou:', len(glob.glob(f'{QUADROS}/*.jpg')), 'quadros')

In [ ]:
# 8) Montagem: 1080x1920, cor, grão de filme, visual de "volta no tempo" e a trilha de efeitos sonoros
%cd /content/mj/videos/trex_corrida
!python3 montar.py {QUADROS} colab/dados/sfx_trex.ogg {SAIDA}/trex_cinema.mp4
from IPython.display import Video
display(Video(f'{SAIDA}/trex_cinema.mp4', embed=True, width=360))
from google.colab import files
files.download(f'{SAIDA}/trex_cinema.mp4')